In [28]:
import numpy as py
import pandas as pd

In [29]:
movies = pd.read_csv('tmdb_5000_movies.csv')
credits = pd.read_csv('tmdb_5000_credits.csv')

In [30]:
movies = movies.merge(credits, on='title')
movies = movies[['movie_id','title','overview','genres','keywords','cast','crew']]

In [31]:
movies.isnull().sum()

movie_id    0
title       0
overview    3
genres      0
keywords    0
cast        0
crew        0
dtype: int64

In [32]:
movies.dropna(inplace=True)
movies.duplicated().sum()

0

In [33]:
movies.iloc[0].genres

'[{"id": 28, "name": "Action"}, {"id": 12, "name": "Adventure"}, {"id": 14, "name": "Fantasy"}, {"id": 878, "name": "Science Fiction"}]'

In [34]:
import ast
def convert(el):
    list1=[]
    for i in ast.literal_eval(el):
        list1.append(i['name'])
    return list1

In [35]:
movies['genres']=movies['genres'].apply(convert)
movies['keywords']=movies['keywords'].apply(convert)

In [36]:
def newcast(el):
    list1=[]
    count=0
    for i in ast.literal_eval(el):
        if count!=3:
         list1.append(i['name'])
         count+=1
        else:
           break
    return list1

In [37]:
movies['cast']=movies['cast'].apply(newcast)

In [38]:
def newcrew(el):
    list1=[]
    for i in ast.literal_eval(el):
        if(i['job']=='Director'):
         list1.append(i['name'])
         break
    return list1

In [39]:
movies['crew']=movies['crew'].apply(newcrew)

In [40]:
movies['overview']=movies['overview'].apply(lambda x:x.split())

In [41]:
movies['genres']=movies['genres'].apply(lambda x:[i.replace(" ","") for i in x])
movies['keywords']=movies['keywords'].apply(lambda x:[i.replace(" ","") for i in x])
movies['cast']=movies['cast'].apply(lambda x:[i.replace(" ","") for i in x])
movies['crew']=movies['crew'].apply(lambda x:[i.replace(" ","") for i in x])

In [42]:
movies['tags']=movies['overview']+movies['genres']+movies['keywords']+movies['cast']+movies['crew']

In [43]:
df= movies[['movie_id','title','tags']]

In [44]:
df['tags']=df['tags'].apply(lambda x:" ".join(x))
df['tags']=df['tags'].apply(lambda x:x.lower())

C:\Users\sabir\AppData\Local\Temp\ipykernel_14240\3156571092.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['tags']=df['tags'].apply(lambda x:" ".join(x))
C:\Users\sabir\AppData\Local\Temp\ipykernel_14240\3156571092.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['tags']=df['tags'].apply(lambda x:x.lower())


In [45]:
from nltk.stem import PorterStemmer
ps=PorterStemmer()

In [46]:
def stem(text):
    l= []
    for i in text.split():
        l.append(ps.stem(i))
    return " ".join(l)

In [47]:
df['tags']= df['tags'].apply(stem)

C:\Users\sabir\AppData\Local\Temp\ipykernel_14240\3201515838.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['tags']= df['tags'].apply(stem)


In [48]:
from sklearn.feature_extraction.text import CountVectorizer
cv= CountVectorizer(max_features=5000, stop_words='english')

In [49]:
vectors= cv.fit_transform(df['tags']).toarray()

In [50]:
from sklearn.metrics.pairwise import cosine_similarity

In [51]:
similarity= cosine_similarity(vectors)

In [52]:
def recommend(movie):
    movie_index= df[df['title']==movie].index[0]
    distance= similarity[movie_index]
    movies_list= sorted(list(enumerate(distance)), reverse=True, key=lambda x:x[1])[1:6]

    for i in movies_list:
        print(df.iloc[i[0]].title)

In [53]:
recommend('Batman Begins')

The Dark Knight
Batman
Batman
The Dark Knight Rises
10th & Wolf


In [54]:
import pickle as pl
pl.dump(df, open('movies.pkl', 'wb'))
pl.dump(similarity, open('similarity.pkl','wb'))